# RMT-PPAD migration NB83 - Phase P4 (LaneSegHead + MTDETRDecoder wiring)

**Purpose.** Verify that building MTDETR from the new lane-only YAML
(`rtdetr-l_bdd_clr_lane.yaml`) succeeds, the resulting model has
`LaneSegHead` as its seg_head (no drivable head anywhere), and an eval
forward pass on a dummy (1, 3, 640, 640) tensor returns without error.

**Acceptance (appendix-path3 sec 7.6):**
- `model.seg_head` is `LaneSegHead`
- `model.seg_head.adapter` is `GCAtoCLRAdapter`
- `model.seg_head.lane_head` is `CLRHeadForSquareImage`
- NO `drivable`-named module anywhere
- Eval forward returns; seg_mask in the output tuple is a dict with key
  `lane_output`

**Wall time:** ~1-2 min (mmcv install + model build + forward).

### Cell 1: Mount Drive, install mmcv

In [1]:
import os, sys, subprocess
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing {REPO_ROOT} -- verify Drive sync.')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# CLRHead needs mmcv.cnn.ConvModule. P3 installed mmcv on a previous
# notebook session, but per the per-notebook isolation rule we install
# again here.
try:
    import mmcv  # noqa: F401
    print(f'[ok] mmcv already installed: {mmcv.__version__}')
except ImportError:
    print('[install] mmcv (~1-3 min wheel build)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mmcv'])
    import mmcv  # noqa: F401
    print(f'[ok] mmcv installed: {mmcv.__version__}')

import torch
print('torch', torch.__version__)
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
[install] mmcv (~1-3 min wheel build)...
[ok] mmcv installed: 2.2.0
torch 2.10.0+cu128
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


### Cell 2: Build the lane-only MTDETR + smoke forward
`verify_lane_head_integration.py` imports `MTDETR` from the VENDORED
`stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/` (NOT from any
pip-installed ultralytics), constructs the model from
`rtdetr-l_bdd_clr_lane.yaml`, runs eval forward, and asserts the
LaneSegHead structure + no-drivable invariant.

In [2]:
import sys, os
VERIFY_SCRIPT = 'stage2/rmt_ppad_migration/P4_lane_head_integration/tools/verify_lane_head_integration.py'
log = os.path.join(LOG_DIR, 'NB83_lane_head_verify.log')
rc = run_streaming([sys.executable, '-u', VERIFY_SCRIPT],
                   log_path=log, check=False)
if rc != 0:
    raise RuntimeError(f'verify_lane_head_integration.py rc={rc}; see {log}')

[run_streaming] command: /usr/bin/python3 -u stage2/rmt_ppad_migration/P4_lane_head_integration/tools/verify_lane_head_integration.py
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/NB83_lane_head_verify.log
[smoke] importing MTDETR from /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[smoke] building model from /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/rmt_ppad_migration/vendor/RMT-PPAD/ultralytics/cfg/models/mt-detr/rtdetr-l_bdd_clr_lane.yaml
WARNING ⚠️ no model scale passed. Assuming scale='l'.
[smoke] built; total params = 35,303,283
[smoke] seg_head at "model.28.seg_head": LaneSegHead

### Cell 3: In-process model build + named_modules dump
Same acceptance test run inside the notebook process so we can dump the
seg_head subtree for the record.

In [3]:
import sys
from pathlib import Path

RMT_PPAD_ROOT = Path(REPO_ROOT) / 'stage2' / 'rmt_ppad_migration' / 'vendor' / 'RMT-PPAD'
YAML_PATH = RMT_PPAD_ROOT / 'ultralytics' / 'cfg' / 'models' / 'mt-detr' / 'rtdetr-l_bdd_clr_lane.yaml'

# Make sure the vendored ultralytics wins over any pip-installed one.
sys.path = [x for x in sys.path if 'ultralytics' not in x.lower()]
sys.path.insert(0, str(RMT_PPAD_ROOT))
for k in list(sys.modules):
    if k == 'ultralytics' or k.startswith('ultralytics.'):
        del sys.modules[k]

from ultralytics import MTDETR
model = MTDETR(str(YAML_PATH))
inner = model.model.cpu().eval()

n_params_total = sum(p.numel() for p in inner.parameters())
print(f'Total params: {n_params_total:,}\n')

print('seg_head subtree:')
for name, m in inner.named_modules():
    if name.endswith('.seg_head') or '.seg_head.' in name:
        depth_dots = name.count('.')
        if depth_dots <= 5:  # don't print every conv leaf
            print(f'  {name:60s} {type(m).__name__}')

# Confirm no drivable.
bad = [name for name, _ in inner.named_modules() if 'drivable' in name.lower()]
assert not bad, f'drivable modules present: {bad}'
print(f'\nNo `drivable` modules in {sum(1 for _ in inner.named_modules())} named modules.')

print('\n[P4 result] PASS')

WARNING ⚠️ no model scale passed. Assuming scale='l'.
Total params: 35,303,283

seg_head subtree:
  model.28.seg_head                                            LaneSegHead
  model.28.seg_head.adapter                                    GCAtoCLRAdapter
  model.28.seg_head.adapter.proj                               ModuleList
  model.28.seg_head.adapter.proj.0                             Sequential
  model.28.seg_head.adapter.proj.1                             Sequential
  model.28.seg_head.adapter.proj.2                             Sequential
  model.28.seg_head.lane_head                                  CLRHeadForSquareImage
  model.28.seg_head.lane_head.prior_embeddings                 Embedding
  model.28.seg_head.lane_head.seg_decoder                      SegDecoder
  model.28.seg_head.lane_head.seg_decoder.dropout              Dropout2d
  model.28.seg_head.lane_head.seg_decoder.conv                 Conv2d
  model.28.seg_head.lane_head.reg_modules                      ModuleList
  m